# 10a — Reliability and observability — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/10.md) · [Course map](../PLAN.md) · [Project](../../projects/stage10/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Define service-level indicators
- Design idempotent writes and safe logs
- Plan timeout and dependency failure behavior


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

Production work includes availability, correct failures, deployment, recovery, and maintenance. Logs describe events, metrics aggregate measurements, and traces connect work across components. Use request IDs and model/prompt/index versions so an answer can be reconstructed. Do not log raw credentials or confidential retrieved text by default.

A health endpoint can report process liveness; readiness checks whether required dependencies are usable. A retry is useful only for a transient failure and can amplify overload. Set connect/read/total deadlines, bounded queues, rate limits, and circuit-breaking behavior. Idempotency avoids duplicate side effects after uncertain responses.

Define an SLO against user-visible outcomes: for example, a chosen fraction of eligible requests succeed below a latency target. An error budget quantifies permitted failures over a window. Backups are only useful after a restore drill. Migrations, rollback, secrets management, and capacity planning belong in the runbook, not in memory.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
event = {"request_id": "example-001", "status": "ok", "latency_ms": 12.5, "model_version": "offline"}
print(event)


## Exercise 1: SLO calculation

Return fraction of events with ok=True and latency_ms<=target. Empty input raises ValueError.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def slo_fraction(events, target=500):
    if not events: raise ValueError("No observations")
    return sum(e["ok"] and e["latency_ms"] <= target for e in events)/len(events)


In [ ]:
assert slo_fraction([{"ok": True, "latency_ms": 100}, {"ok": False, "latency_ms": 10}]) == .5
print("Exercise 1: checks passed")


**Why this works:** A latency-only dashboard can look healthy while every request fails.


## Exercise 2: Log allowlist

Keep request_id,status,latency_ms only; omit every other key.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def safe_log(event):
    return {k: event[k] for k in ("request_id", "status", "latency_ms") if k in event}


In [ ]:
assert safe_log({"request_id": "x", "token": "secret", "prompt": "private"}) == {"request_id": "x"}
print("Exercise 2: checks passed")


**Why this works:** Nested and newly added fields make denylist redaction fragile.


## Exercise 3: Idempotency conflict

Given a mutable cache and key,payload,result, save (payload,result) on first call, return existing result for same payload, reject changed payload.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def remember(cache, key, payload, result):
    if key in cache:
        previous_payload, previous_result = cache[key]
        if previous_payload != payload: raise ValueError("Idempotency conflict")
        return previous_result
    cache[key] = (payload, result)
    return result


In [ ]:
cache = {}
assert remember(cache, "k", {"x": 1}, 3) == 3
assert remember(cache, "k", {"x": 1}, 8) == 3
expect_error(ValueError, lambda: remember(cache, "k", {"x": 2}, 4))
print("Exercise 3: checks passed")


**Why this works:** This in-memory teaching version is not atomic across processes; use a durable transactional store in production.


## Independent transfer

Inject a database outage, model timeout, invalid auth, malformed body, oversized prompt, and corrupt document. Record status, latency, logs, recovery steps, and whether retries could duplicate work.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Liveness versus readiness?

   Liveness asks whether the process is running; readiness asks whether it can serve traffic with required dependencies.

2. What proves a backup works?

   A successful restore followed by integrity and application checks.


## Reading and review

- [Primary reference 1](https://docs.docker.com/compose/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
